# 02 — Text Preprocessing for Sentiment Analysis
**Simple (single-core) & Multi-core (parallel) approaches**

---

In [ ]:
!pip install -q nltk pandas tqdm datasets multiprocess

In [ ]:
import re
import string
import time
import multiprocessing as mp
from functools import partial

import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm.notebook import tqdm

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

print(f"CPUs available: {mp.cpu_count()}")

## 1. Load a Sample Dataset

In [ ]:
from datasets import load_dataset

# SST-2: Stanford Sentiment Treebank (binary sentiment)
ds = load_dataset("glue", "sst2", split="train[:5000]")
df = pd.DataFrame({'text': ds['sentence'], 'label': ds['label']})
print(f"Shape: {df.shape}")
df.head()

## 2. Preprocessing Pipeline — Simple (Single-core)

In [ ]:
def preprocess_text(text: str,
                    remove_stopwords: bool = True,
                    lemmatize: bool = True) -> str:
    """Full preprocessing pipeline for sentiment text."""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # 3. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # 4. Remove digits
    text = re.sub(r'\d+', '', text)
    # 5. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # 6. Tokenise
    tokens = text.split()
    # 7. Remove stopwords
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP_WORDS]
    # 8. Lemmatize
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Quick test
sample = "I absolutely LOVED the 2nd movie! Visit http://example.com for more."
print("Original :", sample)
print("Processed:", preprocess_text(sample))

In [ ]:
# --- Single-core (pandas apply) ---
start = time.time()
df['text_clean_single'] = df['text'].apply(preprocess_text)
single_time = time.time() - start
print(f"Single-core time: {single_time:.2f}s")
df[['text', 'text_clean_single']].head(3)

## 3. Multi-core Preprocessing

In [ ]:
def preprocess_chunk(chunk, **kwargs):
    """Apply preprocessing to a list (chunk) of texts."""
    return [preprocess_text(t, **kwargs) for t in chunk]

def parallel_preprocess(texts, n_workers=None, **kwargs):
    """Split texts across workers and process in parallel."""
    if n_workers is None:
        n_workers = mp.cpu_count()
    
    texts = list(texts)
    chunk_size = max(1, len(texts) // n_workers)
    chunks = [texts[i:i+chunk_size] for i in range(0, len(texts), chunk_size)]
    
    fn = partial(preprocess_chunk, **kwargs)
    with mp.Pool(n_workers) as pool:
        results = pool.map(fn, chunks)
    
    # Flatten
    return [item for sublist in results for item in sublist]

# --- Multi-core ---
start = time.time()
clean_texts = parallel_preprocess(df['text'], n_workers=mp.cpu_count())
multi_time = time.time() - start
df['text_clean_multi'] = clean_texts

print(f"Multi-core time ({mp.cpu_count()} workers): {multi_time:.2f}s")
print(f"Speedup: {single_time / multi_time:.2f}x")

## 4. Parallel Preprocessing with Hugging Face `datasets` (fastest)

In [ ]:
from datasets import Dataset

hf_ds = Dataset.from_pandas(df[['text', 'label']])

def hf_preprocess(batch):
    batch['text_clean'] = [preprocess_text(t) for t in batch['text']]
    return batch

start = time.time()
hf_ds = hf_ds.map(hf_preprocess, batched=True, batch_size=500, num_proc=mp.cpu_count())
hf_time = time.time() - start

print(f"HF datasets map time: {hf_time:.2f}s")
print(hf_ds[:3])

## 5. Timing Comparison

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

methods = ['Single-core\n(pandas apply)', f'Multi-core\n({mp.cpu_count()} workers)', 'HF datasets\n(map+num_proc)']
times   = [single_time, multi_time, hf_time]

colors = ['#E53935', '#1E88E5', '#43A047']
bars = plt.bar(methods, times, color=colors, edgecolor='white', linewidth=1.5)
for bar, t in zip(bars, times):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{t:.2f}s', ha='center', va='bottom', fontweight='bold')
plt.title('Preprocessing Time Comparison (5 000 samples)', fontsize=14)
plt.ylabel('Time (seconds)')
plt.tight_layout()
plt.show()

## 6. Verify Results are Identical

In [ ]:
assert list(df['text_clean_single']) == list(df['text_clean_multi']), "Mismatch!"
print("✓ Single-core and multi-core results are identical.")

# Show a sample
df[['text', 'text_clean_single', 'label']].sample(5, random_state=42)

## 7. Save Preprocessed Data

In [ ]:
output_path = "/tmp/sentiment_preprocessed.csv"
df[['text_clean_single', 'label']].rename(columns={'text_clean_single': 'text'}).to_csv(output_path, index=False)
print(f"Saved to {output_path}")
!head -5 {output_path}

---
**Next Notebook →** `03_inter_annotator_agreement.ipynb`